### v9 修复记录（残留问题修复：CTC同步 + RNG统一 + LR偏移修复）

**审查目标**：修复v8审查中发现的4项残留问题，确保CTC Trainer与MultiHead Trainer完全对齐，消除Python全局RNG与PyTorch RNG的混用，修复LR调度偏移。

**v8残留问题修复**：

| # | 问题 | 严重度 | 影响 | 修复 |
|---|------|--------|------|------|
| 1 | CTC Trainer缺少`_pre_epoch_hook`/独立generator/`worker_init_fn`/`persistent_override` | 🟡 重要 | CTC训练每epoch数据顺序不可控、worker资源泄漏、无epoch级RNG重置 | 同步MultiHead Trainer的全部DataLoader管理机制 |
| 2 | `_train_epoch`中CutMix决策使用`random.random()`（Python全局RNG） | 🟡 重要 | 与PyTorch RNG混用，`num_workers=0`时与数据增强耦合，短路求值改变RNG推进步数 | 改为`t.rand(1).item()` |
| 3 | `_apply_roi_refine`中ROI决策使用`random.random()` | 🟡 重要 | 模型forward路径中使用Python RNG，与训练逻辑耦合 | 改为`t.rand(1).item()`，移除`import random` |
| 4 | `lr_scheduler.step()`条件`epoch > config.start_epoch`导致恢复后首epoch LR偏移1步 | 🟡 重要 | 恢复训练时首epoch使用上一epoch的LR，warmup阶段影响较大 | 改为`epoch > 0` |

**修复#1详解**（CTC Trainer同步MultiHead）：
- 添加`_base_seed=42`和`_train_generator`，`__init__`中创建DataLoader时传入generator
- 添加`_worker_init_fn`：worker进程独立初始化numpy/random种子
- `_make_loader`新增`generator`和`persistent_override`参数，与MultiHead签名一致
- 添加`_pre_epoch_hook`：每epoch重置全局RNG + 重建DataLoader（`persistent_override=False`）
- `_rebuild_dataloaders`传入generator

**修复#2详解**（CutMix RNG统一）：
- 旧代码：`random.random() < config.cutmix_prob` → 使用Python全局RNG
- 新代码：`t.rand(1).item() < config.cutmix_prob` → 使用PyTorch全局RNG
- 与v8添加的`_pre_epoch_hook`全局RNG重置配合，`t.manual_seed(epoch_seed)`会重置PyTorch RNG
- CutMix决策现在与DataLoader shuffle、数据增强使用同一RNG体系

**修复#3详解**（ROI RNG统一）：
- 旧代码：`random.random() < self.roi_gt_prob` → 使用Python全局RNG
- 新代码：`t.rand(1).item() < self.roi_gt_prob` → 使用PyTorch全局RNG
- 移除`models/multihead.py`中的`import random`，消除Python RNG依赖

**修复#4详解**（LR偏移修复）：
- 旧代码：`if epoch > config.start_epoch: step()` → 恢复训练时首epoch不step
- 保存checkpoint时scheduler.last_epoch=N（已step到当前epoch）
- 恢复后epoch=N+1不step → 使用epoch=N的LR（偏移1步）
- 新代码：`if epoch > 0: step()` → 除epoch 0外每个epoch都step
- 全新训练：epoch 0不step（使用初始LR），epoch 1+都step ✓
- 恢复训练：epoch start_epoch也step（推进到正确LR） ✓


### v8 修复记录（训练链路全链路深度审查 + checkpoint恢复/随机状态/CTC同步修复）

**审查目标**：沿训练链路系统性审查所有可能导致acc无法提升的潜在问题与性能瓶颈，重点定位"每轮epoch训练从同一点开始"现象的根本原因。

**v4-v7 修复有效性验证**：

| 版本 | 修复 | 验证结果 | 残留问题 |
|------|------|----------|----------|
| v4#1 | DataLoader独立generator | ✅ 数据顺序每epoch不同 | ⚠️ 仅控制shuffle，不控制`__getitem__`中的random调用 |
| v4#2 | resume_weights_only=False | ✅ notebook中完整恢复 | ❌ baseline.py仍强制True |
| v5#1 | lr_scheduler.step()时机 | ✅ epoch>start_epoch时调用 | ⚠️ 恢复后首epoch LR偏移1步 |
| v5#2 | _pre_epoch_hook清理旧worker | ✅ 资源释放 | - |
| v6#1 | zero_grad条件修复(MultiHead) | ✅ MultiHead已修复 | ❌ CTC Trainer未同步修复 |
| v6#4 | persistent_workers=False | ✅ 重建时传入 | ❌ CTC Trainer未同步 |
| v7#1 | LabelSmoothEntropy size_average='none' | ✅ mask过滤生效 | - |
| v7#2 | attention loss requires_grad | ✅ 极端情况backward不失败 | - |

**v8 新发现与修复**：

| # | 问题 | 严重度 | 影响 | 修复 |
|---|------|--------|------|------|
| 1 | CTC Trainer zero_grad Bug：`(i+1)%grad_accum_steps==1`在grad_accum_steps=1时永远为False | 🔴 **致命** | CTC训练时梯度无限累积，模型完全无法学习 | 改为`i%grad_accum_steps==0`，与MultiHead一致 |
| 2 | baseline.py强制`resume_weights_only=True`，与notebook的`False`不一致 | 🔴 **致命** | 通过脚本运行时optimizer/scheduler/scaler全部重置，start_epoch归零，每次都从同一点开始训练 | 改为`False`，与notebook一致 |
| 3 | MultiHeadTrainer EMA shadow model未从checkpoint恢复 | 🔴 **严重** | 恢复训练时EMA从train_model拷贝而非checkpoint的EMA权重，评估结果不可靠，best model选择可能失效 | 在EMA创建后从`ckpt['model']`加载EMA shadow权重 |
| 4 | CTC Trainer完全忽略`resume_weights_only`标志 | 🔴 **严重** | 无论标志取何值都恢复完整训练状态，行为与用户预期不符 | 添加resume_weights_only分支逻辑 |
| 5 | CTC Trainer始终加载EMA权重到训练模型，不加载`train_model` | 🔴 **严重** | 继续训练时从EMA权重（滑动平均）开始而非训练权重，初期梯度异常 | resume_weights_only=False时优先加载train_model |
| 6 | 旧best checkpoint清理逻辑bug：先更新`best_checkpoint_path`再删除旧文件 | 🟡 重要 | `old_best`拿到的是新路径而非旧路径，旧文件永远不会被删除，磁盘积累多个best文件 | 在更新前保存旧路径引用 |
| 7 | `best_checkpoint_path`未从checkpoint恢复 | 🟡 重要 | 恢复训练后旧best文件无法被清理 | 在两个Trainer的恢复逻辑中添加恢复 |
| 8 | `_pre_epoch_hook`未重置全局Python/NumPy/PyTorch RNG | 🟡 重要 | CutMix决策、ROI refine、数据增强中的`random.random()`共享全局RNG，无法独立复现特定epoch行为 | 在`_pre_epoch_hook`中添加`random.seed(epoch_seed)`/`np.random.seed(epoch_seed)`/`t.manual_seed(epoch_seed)` |

**关键Bug #1 详解**（CTC Trainer zero_grad）：
- 旧代码：`if (i+1) % config.grad_accum_steps == 1: self.optimizer.zero_grad()`
- 当`grad_accum_steps=1`（默认值）：`(i+1)%1 == 0`，`0==1`永远为False
- **zero_grad()永远不会被调用**，梯度无限累积
- 此Bug在v6中已为MultiHead Trainer修复，但CTC Trainer未同步
- 修复：`if i % config.grad_accum_steps == 0: self.optimizer.zero_grad()` ✅

**关键Bug #2 详解**（baseline.py resume_weights_only）：
- 旧代码：`config.resume_weights_only = True`（强制True）
- 通过`python baseline.py`运行时，即使找到checkpoint也不会恢复optimizer/scheduler/scaler
- start_epoch被强制归零，LR回到warmup初始极低值，optimizer动量全部丢失
- 这是"每轮epoch训练从同一点开始"现象的**直接原因之一**
- 修复：`config.resume_weights_only = False`，与notebook一致 ✅

**关键Bug #3 详解**（EMA shadow model未恢复）：
- `save_model`保存`ckpt['model']`（EMA权重）和`ckpt['train_model']`（训练权重）
- `resume_weights_only=False`时，训练模型正确加载了`ckpt['train_model']`
- 但EMA shadow model是从train_model深拷贝创建的，而非从`ckpt['model']`加载
- 结果：EMA shadow = train_model权重 ≠ checkpoint中保存的EMA权重
- 评估使用错误的EMA权重 → best model选择可能失效 → acc无法提升
- 修复：EMA创建后从`ckpt['model']`加载shadow权重 ✅

**关键Bug #8 详解**（全局RNG未重置）：
- `_pre_epoch_hook`仅重建了DataLoader的`torch.Generator`（控制shuffle顺序）
- 但`random.random()`（CutMix决策、ROI refine、数据增强）使用全局Python RNG
- 全局RNG在epoch之间持续推进，不会被重置
- 无法独立复现特定epoch的行为，参数变更会级联影响所有后续epoch
- 修复：在`_pre_epoch_hook`中用`epoch_seed = base_seed + epoch*1000`重置全局RNG ✅

**ipynb JSON格式修复**：
- 修复了source数组末尾的trailing comma（line 347）
- 修复了重复的`]`闭合括号（line 543）
- 现在notebook可以被Jupyter正确解析


# 街景字符识别 - FPN Multi-Head 模型

基于 PyTorch 的街景字符识别项目，采用 FPN + Multi-Head Attention 架构。

**支持三种模型**：`fpn_multihead`（默认）、`transformer`、`ctc`

**支持两种GPU环境**：
- NVIDIA A100 (8核 / 32GB内存 / 24GB显存)
- AMD ROCm (8核 / 200GB内存 / 192GB显存)

运行时自动检测GPU环境并选择对应的参数配置（GPUProfile策略模式）。

**torch.compile 支持**：AMD ROCm 环境下自动启用 `torch.compile`，首次运行需编译 kernel（约5-15分钟），后续运行使用缓存。

**日志路径**：`{项目目录}/logs/train_YYYYMMDD_HHMMSS.log`

### v7 修复记录（训练链路全链路审查 + loss计算Bug修复）

**审查目标**：沿训练链路逐一验证v4-v6所有修复确实有效，系统性探查acc无法提升的剩余瓶颈。

**v4-v6 修复有效性验证**：

| 版本 | 修复 | 验证结果 |
|------|------|----------|
| v4#1 | DataLoader独立generator | ✅ `_pre_epoch_hook`每epoch用`seed=42+epoch`重建generator，数据顺序确实不同 |
| v4#2 | resume_weights_only=False | ✅ 完整恢复optimizer/scheduler/scaler状态 |
| v5#1 | lr_scheduler.step()移到epoch开始前 | ✅ `epoch > start_epoch`时才调用，LR调度与epoch对齐 |
| v5#2 | _pre_epoch_hook清理旧worker | ✅ 调用`_cleanup_dataloader`释放资源 |
| v6#1 | zero_grad条件修复 | ✅ `i%1==0`恒为True，每个batch都正确清零梯度 |
| v6#4 | persistent_workers=False | ✅ 重建时传入`persistent_override=False` |

**v7 新发现与修复**：

| # | 问题 | 严重度 | 影响 | 修复 |
|---|------|--------|------|------|
| 1 | `cls_loss`/`aux_loss`的`valid_mask`过滤完全失效 | 🔴 **致命** | `LabelSmoothEntropy`返回`size_average='mean'`的标量loss，`(scalar * mask).sum()/mask.sum() = scalar`，mask完全不起作用。空位样本的near-zero loss稀释有效样本梯度信号，后面的head（4,5）受影响最严重 | `LabelSmoothEntropy`改为`size_average='none'`返回逐样本loss，mask现在真正过滤空位样本 |
| 2 | `attention_diversity_loss`/`spatial_ordering_loss`初始tensor缺少`requires_grad=True` | 🟡 重要 | 极端情况下所有pair贡献为0时，backward()失败 | 初始tensor添加`requires_grad=True` |

**关键Bug #1 详解**（cls_loss valid_mask过滤失效）：
- 旧代码：`head_loss = LabelSmoothEntropy(pred[h], label[:, h])` → 返回标量（所有样本的均值）
- 然后：`(head_loss * valid_mask).sum() / valid_mask.sum()` = `head_loss * mask.sum() / mask.sum()` = `head_loss`
- **valid_mask完全不起作用**，空位样本的near-zero loss被包含在均值中
- 对于head 5（仅~10%样本有效）：`mean(loss_valid * 0.1 + loss_empty * 0.9)` ≈ `0.1 * mean(loss_valid)`
- 有效样本的梯度信号被稀释10倍，模型几乎无法学习第6位数字
- 修复：`size_average='none'`使`head_loss`变为shape (B,)的逐样本tensor
- 修复后：`(head_loss * valid_mask).sum() / valid_mask.sum()` = 仅有效样本的均值 ✅

### v6 修复记录（训练链路全面深度审查）

**审查目标**：系统性探查所有可能导致模型准确率(acc)无法有效提升的潜在问题与性能瓶颈，重点定位"每轮epoch训练从同一点开始"现象的根本原因。

**v5 修复有效性验证**：

| v5# | 修复内容 | 验证结果 | 残留问题 |
|-----|---------|---------|----------|
| 1 | lr_scheduler.step()移到epoch开始前 | ✅ 有效 | 无 |
| 2 | _pre_epoch_hook清理旧worker | ✅ 有效 | persistent_workers在重建时仍为True |
| 3 | _diagnose_dataloader不预取数据 | ✅ 有效 | 无 |
| 4 | epoch级LR/roi_gt_prob诊断日志 | ✅ 有效 | 无 |

**v6 新发现与修复**：

| # | 问题 | 严重度 | 影响 | 修复 |
|---|------|--------|------|------|
| 1 | `zero_grad`在`grad_accum_steps=1`时从未调用 | 🔴 **致命** | 条件`(i+1)%1==1`恒为False，梯度跨batch累积永不清零。Batch N的梯度=g₀+g₁+...+gₙ，方向被早期batch严重偏置，模型参数虽在更新但方向完全错误 → **acc无法提升的直接根因** | 改为`i%grad_accum_steps==0` |
| 2 | `attention_diversity_loss`返回CPU tensor | 🟡 重要 | `t.tensor(0.0, requires_grad=True)`未指定device，与CUDA tensor相加时设备不匹配 | 添加device参数 |
| 3 | `cls_loss`/`bbox_loss`/`aux_loss`初始化缺少`requires_grad` | 🟡 重要 | 极端情况下（所有head无valid/empty样本）loss为无梯度叶子tensor，backward()失败 | 初始化时添加`requires_grad=True` |
| 4 | `persistent_workers=True`在每epoch重建DataLoader时无效 | 🟡 重要 | DataLoader每epoch重建，persistent_workers无法跨epoch保持worker，反而导致旧worker未及时释放、资源泄漏 | 重建时传入`persistent_override=False` |

**关键Bug #1 详解**（`zero_grad`从未调用）：
- 旧代码：`if (i + 1) % config.grad_accum_steps == 1: self.optimizer.zero_grad()`
- 当`grad_accum_steps=1`时，`(i+1) % 1 == 0`恒成立，条件永远为False
- 结果：梯度从第一个batch开始累积，永远不会被清零
- Batch 0: step(g₀) → Batch 1: step(g₀+g₁) → Batch N: step(g₀+...+gₙ)
- 虽有grad_clip限制幅度，但方向被早期batch主导，模型无法有效学习
- **受影响配置**：GPUProfile(grad_accum_steps=1)、AMDLargeProfile(1)、AMDMI250Profile(1)
- 修复：`if i % config.grad_accum_steps == 0: self.optimizer.zero_grad()`

### v5 修复记录（训练链路全面审查）

**审查目标**：系统性验证 v4 修复是否真正有效，沿训练链路逐一审查所有可能导致 acc 无法提升的瓶颈。

**v4 修复有效性验证**：

| v4# | 修复内容 | 验证结果 | 残留问题 |
|-----|---------|---------|----------|
| 1 | DataLoader独立generator | ✅ 有效 | 无 |
| 2 | resume_weights_only=False | ✅ 有效 | 无 |
| 3 | roi_gt_prob调度统一 | ✅ 有效 | 无 |
| 4 | cutmix_data用torch RNG | ✅ 有效 | 无 |
| 5 | scaler.step()返回None告警 | ✅ 有效 | 无 |
| 6 | zero_grad冗余调用 | ✅ 有效 | 无 |

**v5 新发现与修复**：

| # | 问题 | 严重度 | 影响 | 修复 |
|---|------|--------|------|------|
| 1 | `lr_scheduler.step()`在epoch结束后调用，warmup LR延迟一epoch | 🔴 关键 | epoch 0使用初始LR（warmup第0步），epoch 1才使用warmup第1步LR，整体LR调度偏移 | 移到epoch开始前调用（跳过首个epoch避免重复step） |
| 2 | `_pre_epoch_hook`重建DataLoader时不清理旧worker进程 | 🔴 关键 | persistent_workers=True时旧worker不释放，内存/文件描述符泄漏，Windows上可能共享内存冲突 | 重建前调用_cleanup_dataloader |
| 3 | `_diagnose_dataloader`预取数据消耗generator状态 | 🟡 重要 | __init__中next(iter(train_loader))消耗全局RNG状态，影响后续随机操作的可复现性 | 改为仅记录dataset大小，不实际加载数据 |
| 4 | 缺少epoch级LR/roi_gt_prob诊断日志 | 🟡 重要 | 无法从日志中验证LR调度和ROI策略是否按预期工作 | _pre_epoch_hook添加EPOCH-PRE日志 |

**LR调度修复详解**（问题#1）：
- 旧代码：`train()` → `_train_epoch()` → `lr_scheduler.step()`（epoch结束后）
- 新代码：`train()` → `lr_scheduler.step()` → `_pre_epoch_hook()` → `_train_epoch()`（epoch开始前）
- 跳过首个epoch的step（`epoch > start_epoch`），因为scheduler创建时已自动step一次
- 这确保每个epoch训练时使用的LR与scheduler的last_epoch完全对齐

### v4 修复记录（训练深度审查）

**根因定位：每轮epoch训练从同一点开始**

经全面审查，定位到以下根因链：

| # | 问题 | 严重度 | 影响 |
|---|------|--------|------|
| 1 | DataLoader无独立generator，shuffle依赖全局PyTorch RNG | 🔴 关键 | 每次重跑notebook时全局RNG被set_seed(42)重置，DataLoader shuffle顺序与首次运行完全相同 |
| 2 | resume_weights_only=True默认值，optimizer/scheduler/scaler全部重置 | 🔴 关键 | 每次重跑都从epoch 0开始warmup，LR回到极低值，模型无法有效学习 |
| 3 | _pre_epoch_hook在resume时epoch 0设roi_gt_prob=0.5，epoch 1突跳到1.0 | 🟡 重要 | 训练初期ROI策略不一致，导致特征不稳定 |
| 4 | cutmix_data使用np.random而非torch RNG | 🟡 重要 | 与set_seed的torch.manual_seed不统一，跨平台/跨run不可复现 |
| 5 | scaler.step()返回None时无告警 | 🟡 重要 | inf/nan梯度导致optimizer step被跳过时，模型静默不学习 |
| 6 | optimizer.zero_grad()在accumulation cycle内双重调用 | 🟢 次要 | 冗余操作，不影响正确性 |

**修复措施**：
1. DataLoader使用独立torch.Generator，每epoch用`seed=42+epoch`重新播种 → 确保每epoch数据顺序不同且可复现
2. 训练入口默认`resume_weights_only=False`，完整恢复训练状态 → 避免optimizer/scheduler重置
3. _pre_epoch_hook统一使用正常roi_gt_prob调度，不再因resume_weights_only走特殊分支
4. cutmix_data改用torch.distributions.Beta + torch.randint → 与全局种子管理统一
5. scaler.step()返回None时输出WARNING日志 → 及时发现梯度异常
6. 移除accumulation cycle内冗余的zero_grad调用
7. 添加epoch级诊断日志：param_norm/grad_norm/lr/scaler_scale

### v3 修复记录
- **修复 aux_loss CutMix标签不一致**：CutMix时辅助损失现在正确使用混合标签，消除对抗梯度
- **修复 warmup start_factor**：从0.01提升到0.1，避免AdamW初始LR过低(2e-6)导致训练停滞
- **修复 spatial_ordering_loss空位head**：排序损失现在跳过空位head对，消除噪声梯度
- **添加 worker_init_fn**：每个DataLoader worker独立随机种子，增强数据增强多样性
- **修复 attn_supervision CutMix**：CutMix时跳过注意力监督，避免错误监督信号
- **修复 aux_loss空位head**：辅助损失现在也为空位head提供"预测class 10"梯度
- **修复 OOM恢复scheduler**：OOM恢复不再推进LR scheduler，保持调度同步

In [ ]:
# 环境配置：设置随机种子、检测GPU、打印版本信息
import torch as t
import sys
import os

from utils.seed import set_seed
from config import config, BASE_DIR, data_dir, IS_MODELSCOPE, GPU_PLATFORM, TOTAL_VRAM_GB, IS_NVIDIA, IS_AMD
from utils.platform import get_precision_config, is_nvidia_cuda

# 平台特定精度配置
precision_config = get_precision_config()
if is_nvidia_cuda():
    try:
        t.set_float32_matmul_precision('high')
        print('TF32 matmul precision: enabled')
    except Exception:
        print('TF32 matmul precision: not supported on this GPU')
    try:
        t.backends.cudnn.allow_tf32 = True
        print('CuDNN TF32: enabled')
    except Exception:
        print('CuDNN TF32: not supported')
else:
    print('TF32/CuDNN: platform-specific settings skipped')

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

set_seed(42)

print('=' * 60)
print('环境信息')
print('=' * 60)
print(f'Python: {sys.version}')
print(f'PyTorch: {t.__version__}')
print(f'GPU Platform: {GPU_PLATFORM.upper()}')
print(f'CUDA available: {t.cuda.is_available()}')
if t.cuda.is_available():
    print(f'CUDA version: {t.version.cuda}')
    print(f'GPU: {t.cuda.get_device_name(0)}')
    props = t.cuda.get_device_properties(0)
    vram = getattr(props, 'total_mem', getattr(props, 'total_memory', 0)) / (1024**3)
    print(f'GPU VRAM: {vram:.1f} GB')
print(f'BASE_DIR: {BASE_DIR}')
print(f'ModelScope环境: {IS_MODELSCOPE}')
print(f'Model type: {config.model_type}')
print(f'Batch size: {config.batch_size}')
print(f'Eval batch size: {config.eval_batch_size}')
print(f'Num workers: {config.num_workers}')
print(f'Gradient accumulation: {config.grad_accum_steps}')
print(f'Equivalent batch: {config.batch_size * config.grad_accum_steps}')
print(f'Input size: {config.input_height}x{config.input_width}')
print(f'Use torch.compile: {config.use_torch_compile}')
if config.use_torch_compile:
    print(f'Compile mode: {config.compile_mode}')
    print(f'Compile dynamic: {config.compile_dynamic}')
    print(f'Compile fullgraph: {config.compile_fullgraph}')
print(f'Gradient checkpoint: {config.use_gradient_checkpoint}')
print(f'Optimizer: {config.optimizer_type}')
print(f'Scheduler: {config.scheduler_type}')
print(f'Learning rate: {config.lr}')
print(f'Warmup epochs: {config.warmup_epochs}')
print(f'cls_loss_weight: {config.cls_loss_weight}')
print(f'bbox_loss_weight: {config.bbox_loss_weight}')
print(f'aux_loss_weight: {config.aux_loss_weight}')
print(f'ordering_loss_weight: {config.ordering_loss_weight}')
print(f'attn_supervision_weight: {config.attn_supervision_weight}')
print(f'attn_diversity_weight: {config.attn_diversity_weight}')
print(f'CutMix prob: {config.cutmix_prob}')
print(f'EMA decay: {config.ema_decay}')
print(f'Dropout: {config.dropout}')
print(f'Early stopping patience: {config.early_stopping_patience}')
print('=' * 60)

In [ ]:
# 下载数据集（仅在数据不存在时执行）
from data.download import download_dataset
download_dataset()

## 数据探索

In [ ]:
# 统计训练集、验证集、测试集的图片数量
from glob import glob

train_list = glob(data_dir['train_data'] + '*.png')
val_list = glob(data_dir['val_data'] + '*.png')
test_list = glob(data_dir['test_data'] + '*.png')

print(f'训练集图片数: {len(train_list)}')
print(f'验证集图片数: {len(val_list)}')
print(f'测试集图片数: {len(test_list)}')

In [ ]:
# 分析训练集中数字位数的分布
import json

with open(data_dir['train_label'], 'r') as f:
    marks = json.load(f)

dicts = {}
for img, mark in marks.items():
    n = len(mark['label'])
    dicts[n] = dicts.get(n, 0) + 1
for k, v in sorted(dicts.items()):
    print(f'{k}位数字的图片数目: {v}')

## 模型定义

创建模型并统计参数量

In [ ]:
# 创建模型并统计参数量
from models import create_model

model = create_model('fpn_multihead')
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'模型类型: fpn_multihead')
print(f'总参数量: {total_params:,}')
print(f'可训练参数量: {trainable_params:,}')
print(f'模型大小: {total_params * 4 / (1024**2):.1f} MB (FP32)')

## 训练 FPN Multi-Head 模型

训练过程中日志会自动记录到 `logs/` 目录下，包含：
- 每个batch的loss、学习率、精度、GPU/CPU内存使用
- 每个epoch的训练/验证精度（joint acc + char acc）、耗时、early stopping状态
- v5新增：每个epoch开始前的LR、roi_gt_prob、generator_seed诊断日志

**torch.compile warmup 说明**：
- AMD ROCm 环境下自动启用 `torch.compile(mode='default', dynamic=True)`
- 首次运行需编译 CUDA kernel，warmup 阶段约需 **5-15 分钟**（取决于缓存状态）
- 后续运行使用 inductor_cache 缓存，warmup 时间大幅缩短
- 编译失败时自动 fallback 到 eager 模式，不影响训练

**v7 修复说明**：
- **致命Bug修复**：`cls_loss`/`aux_loss`的`valid_mask`过滤完全失效——`LabelSmoothEntropy`返回标量loss，`(scalar*mask).sum()/mask.sum()=scalar`，mask不起作用。空位样本near-zero loss稀释有效样本梯度，后面head受影响最严重
- 修复：`LabelSmoothEntropy`改为`size_average='none'`返回逐样本loss(B,)，mask现在真正过滤空位样本
- `attention_diversity_loss`/`spatial_ordering_loss`初始tensor添加`requires_grad=True`

**v6 修复说明**：
- **致命Bug修复**：`zero_grad`在`grad_accum_steps=1`时从未被调用（条件`(i+1)%1==1`恒为False）→ 改为`i%grad_accum_steps==0`
- `attention_diversity_loss`返回tensor现在正确指定device，避免CPU/CUDA设备不匹配
- `cls_loss`/`bbox_loss`/`aux_loss`初始化添加`requires_grad=True`，防止极端情况下backward()失败
- DataLoader每epoch重建时使用`persistent_workers=False`，避免旧worker资源泄漏


In [ ]:
# 训练FPN Multi-Head模型
# 日志自动保存到 {项目目录}/logs/ 目录
#
# resume_weights_only=False (v4默认): 完整恢复optimizer/scheduler/scaler状态继续训练
#   - 适用于中断后原参数继续训练
#   - 加载train_model权重（非EMA），恢复完整训练状态
# resume_weights_only=True: 只加载模型权重，用新optimizer从头训练
#   - 适用于切换优化器/调度器/超参数后重新训练
#   - 加载EMA权重到训练模型，optimizer/scheduler/scaler全部重新初始化
from trainer.multihead import MultiHeadTrainer
from utils.misc import find_latest_checkpoint

latest_checkpoint = find_latest_checkpoint(config.checkpoints)
if latest_checkpoint:
    print(f'发现检查点: {latest_checkpoint}')
    config.pretrained = latest_checkpoint
    config.resume_weights_only = False
    # 如需只加载权重重新训练，改为: config.resume_weights_only = True

trainer = MultiHeadTrainer(model_type='fpn_multihead')
trainer.train()

## 训练诊断（v7增强）

验证数据加载是否每epoch不同，LR调度是否正确，模型参数是否在更新，zero_grad是否正确执行，以及cls_loss的valid_mask过滤是否真正生效。

In [ ]:
# 诊断：验证DataLoader每epoch数据顺序不同
import torch as t
from utils.seed import make_epoch_generator

print('=== DataLoader Shuffle 诊断 ===')
print(f'训练集大小: {len(trainer.train_set)}')
print(f'Batch size: {config.batch_size}')

first_indices = []
for epoch in range(3):
    gen = make_epoch_generator(42, epoch=epoch)
    sampler = t.utils.data.RandomSampler(trainer.train_set, generator=gen)
    indices = list(sampler)[:config.batch_size]
    first_indices.append(indices)
    print(f'Epoch {epoch}: 前{config.batch_size}个样本索引 = {indices[:5]}...')

if first_indices[0] == first_indices[1]:
    print('⚠️ 警告: epoch 0 和 epoch 1 的数据顺序相同！')
else:
    print('✅ 每epoch数据顺序不同，shuffle正常')

In [ ]:
# 诊断：验证LR调度是否正确（v5新增）
print('=== LR Scheduler 诊断 ===')
print(f'Scheduler type: {config.scheduler_type}')
print(f'Base LR: {config.lr}')
print(f'Backbone LR factor: {config.backbone_lr_factor}')
print(f'Warmup epochs: {config.warmup_epochs}')
print(f'Scheduler last_epoch: {trainer.lr_scheduler.last_epoch}')
print(f'Current backbone LR: {trainer.optimizer.param_groups[0]["lr"]:.8f}')
print(f'Current head LR: {trainer.optimizer.param_groups[1]["lr"]:.8f}')
print()
print('LR schedule (first 10 epochs):')
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
test_opt = t.optim.SGD([{'params': [t.randn(1)], 'lr': config.lr * config.backbone_lr_factor},
                         {'params': [t.randn(1)], 'lr': config.lr}], momentum=0.9)
warmup = LinearLR(test_opt, start_factor=0.1, total_iters=config.warmup_epochs)
cosine = CosineAnnealingLR(test_opt, T_max=config.epoches - config.warmup_epochs, eta_min=1e-6)
test_sched = SequentialLR(test_opt, schedulers=[warmup, cosine], milestones=[config.warmup_epochs])
for e in range(min(10, config.epoches)):
    if e > 0:
        test_sched.step()
    print(f'  Epoch {e+1}: backbone_lr={test_opt.param_groups[0]["lr"]:.8f}, head_lr={test_opt.param_groups[1]["lr"]:.8f}')

In [ ]:
# 诊断：验证模型参数在训练中是否更新
raw = trainer._get_raw_model()
first_param = next(iter(raw.parameters()))
print(f'模型首个参数 shape: {first_param.shape}')
print(f'首个参数前5个值: {first_param.data.flatten()[:5].tolist()}')
print(f'参数范数: {first_param.data.norm().item():.4f}')
print()
total_norm = sum(p.data.norm().item() ** 2 for p in raw.parameters()) ** 0.5
print(f'模型总参数范数: {total_norm:.2f}')
print()
if trainer.train_log:
    print('最近5条训练日志:')
    for entry in trainer.train_log[-5:]:
        print(f'  Epoch {entry["epoch"]}: train_joint={entry["train_joint_acc"]:.2f}, '
              f'val_joint={entry["val_joint_acc"]:.2f}, lr={entry["lr"]:.6f}')

In [ ]:
# 诊断（v6新增）：验证zero_grad在grad_accum_steps=1时正确执行
print('=== zero_grad 诊断 (v6关键Bug验证) ===')
print(f'grad_accum_steps = {config.grad_accum_steps}')

# 验证条件逻辑
for steps in [1, 2, 4]:
    zero_grad_batches = [i for i in range(10) if i % steps == 0]
    step_batches = [i for i in range(10) if (i + 1) % steps == 0 or (i + 1) == 10]
    print(f'  grad_accum_steps={steps}: zero_grad at batch {zero_grad_batches}, step at batch {step_batches}')

# 验证旧条件（Bug）vs 新条件（修复）
print()
print('旧条件 (i+1)%grad_accum_steps==1 在 grad_accum_steps=1 时:')
old_trigger = [(i, (i+1)%1==1) for i in range(5)]
print(f'  {old_trigger} → zero_grad永远不会触发！')
print()
print('新条件 i%grad_accum_steps==0 在 grad_accum_steps=1 时:')
new_trigger = [(i, i%1==0) for i in range(5)]
print(f'  {new_trigger} → zero_grad每个batch都正确触发 ✅')

In [ ]:
# 诊断（v7新增）：验证cls_loss的valid_mask过滤是否真正生效
import torch as t
import torch.nn.functional as F
from losses.classification import LabelSmoothEntropy

print('=== cls_loss valid_mask 诊断 (v7关键Bug验证) ===')

B, C = 8, 11
pred = t.randn(B, C)
label = t.tensor([3, 7, 0, 10, 10, 10, 5, 10])
true_lengths = t.tensor([4, 3, 1, 2, 1, 1, 6, 1])

criteria_mean = LabelSmoothEntropy(smooth=0.1, size_average='mean')
criteria_none = LabelSmoothEntropy(smooth=0.1, size_average='none')

for h in range(6):
    valid_mask = (true_lengths > h).float()
    empty_mask = 1.0 - valid_mask
    n_valid = int(valid_mask.sum().item())
    n_empty = int(empty_mask.sum().item())
    
    head_loss_scalar = criteria_mean(pred, label)  # 旧: 标量
    head_loss_per_sample = criteria_none(pred, label)  # 新: 逐样本
    
    old_result = (head_loss_scalar * valid_mask).sum() / valid_mask.sum() if n_valid > 0 else 0
    new_result = (head_loss_per_sample * valid_mask).sum() / valid_mask.sum() if n_valid > 0 else 0
    
    print(f'Head {h}: valid={n_valid}, empty={n_empty}')
    print(f'  旧(标量*mask no-op): {old_result.item():.6f}')
    print(f'  新(逐样本*mask过滤): {new_result.item():.6f}')
    if n_valid > 0 and n_empty > 0:
        ratio = new_result.item() / old_result.item() if old_result.item() != 0 else float('inf')
        print(f'  新/旧比值: {ratio:.2f}x (空位越多比值越大)')
    print()

## 评估

In [ ]:
# 在验证集上评估模型精度（返回joint acc）
val_acc = trainer._eval()
print(f'最佳验证精度: {trainer.best_acc * 100:.2f}%')

In [ ]:
# 逐Head评估每个数字位的分类精度
trainer.eval_detailed()

## torch.compile 审计（可选）

运行系统性编译审计，覆盖编译模式、输入数据类型/尺寸/批次、warmup策略对比、正确性验证等。
完整审计约需 1-2 小时，可按需选择单项审计。

In [ ]:
# 运行 torch.compile 系统性审计（可选，取消注释执行）
# !python scripts/compile_audit.py --audit warmup  # 仅测试warmup策略
# !python scripts/compile_audit.py --audit modes   # 仅测试编译模式
# !python scripts/compile_audit.py --audit all     # 完整审计

## 推理与提交

In [ ]:
# 推理（use_compile=True 时启用 torch.compile 加速推理）
from inference.predict import predicts, ctc_predict, ensemble_predict

# model_path = 'checkpoints/best-resnet101-acc-xx.xx.pth'
# results = predicts(model_path, 'submit.csv', use_tta=True, use_compile=True)

In [ ]:
# TTA 评估
# tta_acc = trainer.eval_tta()
# print(f'TTA Accuracy: {tta_acc * 100:.2f}%')

## 训练日志

In [ ]:
# 查看训练日志
for entry in trainer.train_log[-5:]:
    print(entry)

In [ ]:
# 保存模型
# trainer.save_model('checkpoints/manual_save.pth', save_opt=True)